# Train the wound models on Kaggle

Full-size versions of what runs on the Mac: the **wound outline** (SegFormer-B2, 512 px) and the **wound type**
(ConvNeXt-Tiny, 384 px). Both datasets are public on GitHub, so this notebook fetches everything itself.

**Before you press Run all:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **on**, Persistence
**Files only** (so a timed-out session can resume). One full run takes about 1.5–2 hours of GPU time.

At the end, download `outputs/` (Output tab) and copy the `.pt` files into `wound-ai/checkpoints/` on your
machine, or upload them to your private Hugging Face model repo (see `docs/deploy.md`).

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data

FUSeg (1,010 foot-ulcer photos with outlines) and AZH (930 photos labelled by wound type). Both are public
research datasets; check their terms before any commercial use. To add more outline data later (for example the
Mendeley lower-limb set as a private Kaggle dataset), add another `--pairs name:images_dir:masks_dir`.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
mkdir -p data/raw && cd data/raw
if [ ! -d fuseg ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git src
  (cd src && git sparse-checkout set --no-cone "/data/Foot Ulcer Segmentation Challenge/train/" "/data/Foot Ulcer Segmentation Challenge/validation/")
  mv "src/data/Foot Ulcer Segmentation Challenge" fuseg && rm -rf src
fi
if [ ! -d azh ]; then
  mkdir azh && cd azh
  for f in Train Test; do
    curl -sL -o $f.zip https://raw.githubusercontent.com/uwm-bigdata/wound-classification-using-images-and-locations/main/dataset/$f.zip
    unzip -q $f.zip && rm $f.zip
  done
  rm -rf __MACOSX && cd ..
fi
ls fuseg/train/images | wc -l; ls azh/Train

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
# Same seed and sources as on the Mac, so the splits match and test results are comparable.
python scripts/prepare_data.py \
  --pairs "fuseg:data/raw/fuseg/train/images:data/raw/fuseg/train/labels" \
          "fuseg:data/raw/fuseg/validation/images:data/raw/fuseg/validation/labels" \
  --out data/manifest.csv
python scripts/prepare_data.py \
  --classes "azh:data/raw/azh/Train:wound_type:configs/azh_map.json" "azh:data/raw/azh/Test:wound_type:configs/azh_map.json" \
  --out data/manifest_cls.csv

## 2. Wound outline (about 60–90 minutes)

If the session ends, run this cell again: `--resume` continues from the last finished epoch.

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/train_seg.py --manifest data/manifest.csv --task boundary \
  --arch segformer --encoder mit_b2 --size 512 --batch-size 8 --epochs 60 --patience 12 \
  --out /kaggle/working/runs/boundary --resume

## 3. Wound type (about 15–25 minutes)

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/train_cls.py --manifest data/manifest_cls.csv --target wound_type \
  --backbone convnext_tiny.fb_in22k --size 384 --batch-size 32 --epochs 40 \
  --out /kaggle/working/runs/wound_type

## 4. Test once, then package

The test split is touched once per model version: tune only on validation.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
mkdir -p /kaggle/working/outputs checkpoints
cp /kaggle/working/runs/boundary/boundary.pt checkpoints/
cp /kaggle/working/runs/wound_type/wound_type.pt checkpoints/
python scripts/evaluate.py --manifest data/manifest.csv --ckpt-dir checkpoints --group-cols source --out /kaggle/working/outputs/eval_boundary.json
python scripts/evaluate.py --manifest data/manifest_cls.csv --ckpt-dir checkpoints --group-cols source --use-gt-mask --out /kaggle/working/outputs/eval_wound_type.json || true
cp checkpoints/*.pt /kaggle/working/outputs/
cp /kaggle/working/runs/boundary/history.csv /kaggle/working/outputs/history_boundary.csv
cp /kaggle/working/runs/wound_type/*.json /kaggle/working/outputs/ 2>/dev/null || true
ls -la /kaggle/working/outputs